On test les Grad-CAM sur nos réseaux (https://arxiv.org/abs/1610.02391)

In [1]:
from retinotopy import *
from gradcam import *

Running on metal mps


In [2]:
def logpolar_to_cartesian_mesh_with_padding(logpolar_heatmap, cartesian_shape, size_ratio, rs_max):
    """
    Converts a log-polar heatmap to its Cartesian representation using mesh-based mapping,
    with circular padding to smooth the resulting grid.

    Args:
        logpolar_heatmap (ndarray): 2D array representing the heatmap in log-polar coordinates.
        cartesian_shape (tuple): Dimensions of the output Cartesian grid (height, width).
        size_ratio (float): Minimum radius ratio in the log-polar referential.
        rs_max (float): Maximum radius in log-polar normalized coordinates.

    Returns:
        ndarray: Cartesian representation of the heatmap.
    """
    # Dimensions of the log-polar heatmap
    lp_height, lp_width = logpolar_heatmap.shape

    # Add circular padding to the log-polar heatmap along the angular dimension
    logpolar_heatmap_padded = np.hstack([logpolar_heatmap, logpolar_heatmap[:, :1]])

    # Updated dimensions after padding
    lp_width_padded = lp_width + 1

    # Cartesian grid (output grid)
    cartesian_height, cartesian_width = cartesian_shape
    x_cartesian = np.linspace(-1, 1, cartesian_width)
    y_cartesian = np.linspace(-1, 1, cartesian_height)
    X_cartesian, Y_cartesian = np.meshgrid(x_cartesian, y_cartesian)

    # Compute polar coordinates for the Cartesian grid
    R_cartesian = np.sqrt(X_cartesian**2 + Y_cartesian**2)  # Radius
    Theta_cartesian = np.arctan2(Y_cartesian, X_cartesian)  # Angle (in radians)

    # Normalize radius to match the log-polar grid scaling
    start = np.log2(size_ratio)
    R_cartesian = np.clip(R_cartesian, 0, 1)  # Clamp radius to [0, 1]
    R_logpolar = (np.log2(R_cartesian + 1e-10) - start) / (rs_max - start) * (lp_height - 1)

    # Normalize angle to match the log-polar grid scaling
    Theta_logpolar = (Theta_cartesian % (2 * np.pi)) / (2 * np.pi) * (lp_width_padded - 1)

    # Map the log-polar heatmap onto the Cartesian grid using bilinear interpolation
    R_indices = np.clip(R_logpolar, 0, lp_height - 1)
    Theta_indices = np.clip(Theta_logpolar, 0, lp_width_padded - 1)

    # Perform bilinear sampling
    r0 = np.floor(R_indices).astype(int)
    r1 = np.clip(r0 + 1, 0, lp_height - 1)
    t0 = np.floor(Theta_indices).astype(int)
    t1 = np.clip(t0 + 1, 0, lp_width_padded - 1)

    # Interpolation weights
    w_r1 = R_indices - r0
    w_r0 = 1 - w_r1
    w_t1 = Theta_indices - t0
    w_t0 = 1 - w_t1

    # Interpolated values
    cartesian_heatmap = (
        logpolar_heatmap_padded[r0, t0] * w_r0 * w_t0 +
        logpolar_heatmap_padded[r1, t0] * w_r1 * w_t0 +
        logpolar_heatmap_padded[r0, t1] * w_r0 * w_t1 +
        logpolar_heatmap_padded[r1, t1] * w_r1 * w_t1
    )

    return cartesian_heatmap

In [3]:
# The dataset to import images from

data_set_type = 'full'
args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['val'] # type of images to use
args.resolution = (7,7)
args.do_polar = True
args.do_mask = False
image_datasets = image_datasets_transforms(args, verbose=False)['val']

exp_name = f'_complete_gradcam.parquet'

In [4]:
print('Lets go !')
hash_name = '\\' if platform.uname()[0] == 'Windows' else '/'
#for model_data_set_type in data_set_types:
for model_data_set_type in ['full', 'bbox', 'focus']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')

        print(f'{args.do_polar=}')

        print(50*'.')
        
        model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, args.do_polar) + '.pt'

        model_filename = None if model_data_set_type == 'raw' else model_filename
        
        model = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
        
        annotations = get_annotation('csv')

        df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, args.do_polar) + f'_complete_gradcam.parquet'
        print(df_filename)
        
        if os.path.isfile(df_filename):
            continue
        else:
            df_grad = None
            
            for i_image, (images, label) in tqdm(enumerate(image_datasets)):
            
                image_name = image_datasets.samples[i_image][0].split(hash_name)[-1].split('.')[0]
                ground_true_indices, three_points, ground_true, origin_size = get_ground_true(args, image_name, annotations, 'Imagenet')
                
                if ground_true.min() == 0.0:
                            
                    images = images.to(device)
                    
                    since_grad = time.time()
                    
                    GradCam, pred = get_Grad_cam(model, images.unsqueeze(0), label)
        
                    elapsed_time_grad = time.time() - since_grad
        
                    arg_max_prior = torch.argmax(GradCam).item()
                    
                    GradCam = logpolar_to_cartesian_mesh_with_padding(GradCam, GradCam.shape, 1, args.rs_max)
                    GradCam = torch.tensor(GradCam.reshape(args.resolution[0]*args.resolution[1]))
        
                    
                    position_prior = (arg_max_prior%args.resolution[0], arg_max_prior//args.resolution[1])
        
                    
                    grad_out = th_delete(GradCam, ground_true_indices[0])
            
                    grad_out_max = torch.max(grad_out).item()
                    grad_in_max = torch.max(GradCam[ground_true_indices[0]]).item()
                
                    grad_out_mean = torch.mean(grad_out).item()
                    grad_in_mean = torch.mean(GradCam[ground_true_indices[0]]).item()
        
                    
                    Iou = get_IoU(GradCam, ground_true.reshape(args.resolution[0]*args.resolution[1]))
        
                    
        
                    PG = 1 if grad_in_max > grad_out_max else 0
                    
                    
                    df_grad_ = pd.DataFrame({'image_name':image_name, 'grad_in_max': grad_in_max, 'grad_out_max': grad_out_max,
                                             'grad_in_mean': grad_in_mean, 'grad_out_mean': grad_out_mean, 'position_prior':[position_prior],
                                             'Iou':[Iou], 'PG':PG, 
                                             'pred':pred, 'time_grad':elapsed_time_grad, 'label':label, 'GradCam':[np.asarray(GradCam)]})
                
                    df_grad = store_pandas(df_grad, df_grad_)
            
            df_grad.to_parquet(df_filename)
            model.cpu()

Lets go !
model_data_set_type='full'
..................................................
args.do_polar=True
..................................................
loading .... cached_data/2025-01-05_full_resnet18_retino.pt
cached_data/2025-01-05_full_resnet18_retino_complete_gradcam.parquet


0it [00:00, ?it/s]


KeyError: 'origin_size'

In [ ]:
def get_pandas_cam(model_data_set_type, model_name, exp_name):
    
    df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, True) + exp_name
    print(f'{df_filename=}')
    with open(df_filename, 'r') as csv_file:
        df = pd.read_parquet(df_filename)
        
    return df

In [ ]:
#for model_data_set_type in data_set_types: # data_set_types
for model_data_set_type in ['full', 'bbox', 'focus']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
    #for model_name in ['resnet18']:
        print(f'{model_name=}')    
        print(50*'.')


        
        args.do_polar = False
        print(f'{args.do_polar=}')
        print(50*'.')

        
        df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, True) + exp_name
        results = pd.read_parquet(df_filename)
        
        print(results['grad_in_mean'].mean(), 'grad_in_mean')
        print(results['grad_out_mean'].mean(), 'grad_out_mean')
        print(results['grad_in_mean'].mean()/results['grad_out_mean'].mean(), 'likelihood ratio')
        print(results['grad_in_max'].mean(), 'grad_in_max')
        print(results['grad_out_max'].mean(), 'grad_out_max')
        

        mean_Iou = read_IoU(results['Iou'])
        print(mean_Iou[0], 'mean IoU')

        best_Iou = get_best_Iou(results['Iou'], np.argmax(mean_Iou[0]))
        
        for test_tresh in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
            inpel = 0
            for i, j in enumerate(results['grad_in_max']):
                #print(i, j, results['max_in_heat'][i] , '>', results['max_out_heat'][i])
                if best_Iou[i] > test_tresh and results['label'][i] == results['pred'][i]:
                    inpel += 1
            print(inpel/len(results['grad_in_max']), f'GT-known : {test_tresh}')
    
        
        
        print(results['PG'].mean(), 'PG')
        


In [ ]:
#subplotpars = matplotlib.figure.SubplotParams(left=0.1, right=.95, bottom=0.25, top=.975, hspace=.6)
#data_set_types.append('raw')
for model_data_set_type in ['full', 'bbox', 'focus']:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    df_list = []
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        
        df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, True) + exp_name
        df_list.append(pd.read_parquet(df_filename))


    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    plt.tick_params(axis='both', which='major', labelsize=10)
    
    for df_, label in zip(df_list, ['resnet18', 'resnet50', 'resnet101']):
    
        mean_Iou, list_Iou = read_IoU(df_['Iou'])
    
    
        ax.plot(np.linspace(0, 1, 36), list_Iou, lw=2, marker='.', label=label)
        ax.set_xlabel(f"IoU Threshold", size=12)
        ax.set_ylabel(f"IoU ", size=12)
        ax.spines['left'].set_position(('axes', -0.01))
        #ax.set_yscale("logit", one_half="1/2", use_overline=True)
        ax.grid(which='both')
        ax.legend()
        for side in ['top', 'right'] :ax.spines[side].set_visible(False)
        ax.set_title(f'Average IoU with different IoU threshold', size=10);

In [ ]:
for model_data_set_type in ['full', 'bbox', 'focus']:
   
    df_list = []
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        
        df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, True) + exp_name
        df_list.append(pd.read_parquet(df_filename))


    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    plt.tick_params(axis='both', which='major', labelsize=10)
    GT = {}
    for df_, label in zip(df_list, ['resnet18', 'resnet50', 'resnet101']):
    
        GT[label] = []
        for test_tresh in np.linspace(0, 0.9, 10):
            inpel = 0
            for i, j in enumerate(df_['grad_in_max']):
                if best_Iou[i] > test_tresh and df_['label'][i] == df_['pred'][i]:
                    inpel += 1
            GT[label].append(inpel/len(df_['grad_in_max']))

        ax.plot(np.asarray(np.linspace(0, 0.9, 10)), GT[label], lw=2, marker='.', label=label)
        ax.set_xlabel(f"GT Threshold", size=12)
        ax.set_ylabel(f"GT-known ", size=12)
        ax.spines['left'].set_position(('axes', -0.01))
        ax.set_yscale("logit", one_half="1/2", use_overline=True)
        ax.grid(which='both')
        ax.legend()
        for side in ['top', 'right'] :ax.spines[side].set_visible(False)